# Lab 02 · RAG: ground answers in retrieved knowledge

**Prepared by Mustafa Assaf** · Lecture 1: Ask, check, retrieve · 15–20 minute lab

## Retrieve lecture evidence before answering
**So far:** Lab 01 put a short readiness rule directly in the prompt.

**New:** retrieval-augmented generation (RAG). We store short original summaries of the lecture's data patterns, embed them and retrieve the relevant slide note. Students then check the answer against the named slide.

The local `all-MiniLM-L6-v2` model turns text into 384-number vectors. Chroma stores and searches those vectors on CPU; the first run downloads the model. No model key is needed for retrieval; generation uses the course client.

**At a glance**

![Lab 02 at a glance: Question, then Embed (MiniLM vectors), then Nearest lecture notes (Chroma), then close?, then Evidence-based reply (cites (M28)), then Citation check (cited ⊆ retrieved).](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/02.svg)

## Review your microservices lectures

- **Microservices & Their Patterns**, slides 24, 27, 28, 30.
- **Microservices Lecture 2 - Condensed DDD**, slides 52, 55.

**Recall:** Database per service; API composition vs CQRS; Idempotent consumers and transactional outbox.

Before running the example:

1. Why is CQRS a better fit for searchable order history than a large in-memory API join?
2. Does a valid slide citation prove that an answer is supported?

Slide numbers count from the title slide, including appendices. The small scenarios and code are teaching adaptations, not production services. Use the lecture decks supplied in class; they are not uploaded with this course.

## The idea in context

Retrieval-augmented generation puts selected external evidence into a prompt. Embeddings find semantic neighbours; neighbours still need relevance checks.

Citing a valid source ID does not by itself prove factual support.

| Key term | Meaning |
| --- | --- |
| **Context window** | The maximum amount of text a model can read in one call. |
| **Embedding** | A vector of numbers that represents the meaning of a text. |
| **Vector database** | A store that finds the vectors nearest to a query vector. |
| **Grounded answer** | An answer supported by, and citing, the retrieved sources. |

## Pattern · Retrieval-Augmented Generation

*Single-agent pattern from the [Architectural Patterns for Agentic AI](https://karthikv1392.github.io/agentic-patterns/) field guide.*

| | |
| --- | --- |
| **Context** | Useful answers often depend on private, recent, or domain-specific material that the model's weights do not reliably hold. |
| **Problem** | Without evidence, a model may hallucinate, miss local facts, or answer from outdated knowledge. |
| **Solution** | Search a knowledge base for relevant pieces, place them beside the question, and instruct the model to ground its answer in them. |
| **Use when** | Answers depend on a document collection; citations or provenance matter; knowledge changes independently of the model. |
| **Trade-off** | Answer quality now depends on retrieval quality, chunking, and the context budget. |
| **Flow** | Question → Retrieve → Augment → Grounded answer |
| **Quality attributes** | Benefits: Reliability, Data freshness, Modifiability, Cost, Privacy. Liabilities: Latency, Maintainability, Reliability, Data freshness. |
| **In this lab** | Embeddings and Chroma retrieve lecture notes with slide IDs. A threshold supports abstention. Code checks citation validity; students inspect whether the source actually supports the claim. |

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Practice answers are scripted. For a live model, choose `PROVIDER` (`"freeinference"`, `"gemini"`, or `"deepseek"`), save its key in Colab Secrets (`FREEINFERENCE_API_KEY`, `GEMINI_API_KEY`, or `DEEPSEEK_API_KEY`), and change `MODE` to `"live"`. `ask` stops after 12 calls; run this cell again to reset the count.

In [ ]:
from importlib.util import find_spec
if not all(find_spec(name) for name in ['coursekit', 'chromadb']):
    %pip install -q "agentic-ai-coursekit[rag] @ https://github.com/mkassaf/agentic-ai-course/archive/9c78f2181b0f793c60b2b081c79ae74276026e7d.zip"
from coursekit import make_ask
MODE = "practice"
PROVIDER = "freeinference"  # or "gemini" or "deepseek"
ask = make_ask(mode=MODE, provider=PROVIDER)

## Step 1 · Store slide notes with provenance
These summaries distinguish database ownership, small lookup composition, searchable read models, duplicate events and atomic publication intent. They do not reproduce the full decks.

In [ ]:
notes = {
    "M24": "Database per service: each service owns private data; other services use APIs or events, not its tables.",
    "M27": "API composition joins a few services in the caller for a small lookup by id; large searches are a poor fit.",
    "M28": "CQRS maintains a separate read model for searchable order history; the view can lag behind writes.",
    "D52": "At-least-once delivery can repeat an event; store the processed eventId and the business effect in the same local DB transaction.",
    "D55": "Transactional outbox commits business state and publication intent in one local transaction; relay retries may duplicate.",
}
question = "Support needs searchable order history across services. Which data pattern fits, and what is its cost?"
for sid, text in notes.items():
    print(sid, text)

## Step 2 · Compare semantic vectors
Cosine similarity compares the directions of two vectors. It is a retrieval heuristic, not proof that a note answers the question.

In [ ]:
import numpy as np
from chromadb.utils.embedding_functions import DefaultEmbeddingFunction
embed = DefaultEmbeddingFunction()
q_vec, query_note, ownership_note = embed([question, notes["M28"], notes["M24"]])
def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))
print("Vector size:", len(q_vec))
print("Question vs CQRS:", round(cosine(q_vec, query_note), 2))
print("Question vs ownership:", round(cosine(q_vec, ownership_note), 2))

## Step 3 · Search a vector collection
A smaller cosine distance means a nearer match. We retain the slide ID as the source identifier. This is an in-memory teaching store, not persistent course infrastructure.

In [ ]:
import chromadb
from chromadb.config import Settings
collection = chromadb.Client(Settings(anonymized_telemetry=False)).get_or_create_collection(
    "lecture-notes", embedding_function=embed, configuration={"hnsw": {"space": "cosine"}})
collection.upsert(ids=list(notes), documents=list(notes.values()))
found = collection.query(query_texts=[question], n_results=3)
for sid, distance in zip(found["ids"][0], found["distances"][0]):
    print(sid, round(distance, 2))

## Step 4 · Retrieve or abstain
The threshold is a small classroom heuristic that must be tuned on real questions. Unrelated questions should not reach the generator. Inspect distances before trusting it.

In [ ]:
def retrieve(question, max_distance=0.7):
    found = collection.query(query_texts=[question], n_results=1)
    return {sid: notes[sid] for sid, d in zip(found["ids"][0], found["distances"][0]) if d <= max_distance}
questions = [question, "May Reporting read the private tables owned by Order?", "What should we eat for lunch?"]
for q in questions:
    print(list(retrieve(q)), q)

## Step 5 · Answer with a citation, then check support
The code checks **citation validity**: a cited ID must have been retrieved. That alone does not prove the source supports the claim. Students must inspect that semantic support and the stated trade-off.

In [ ]:
import re
scripted = ["Use a CQRS read model for searchable history; it may lag behind writes [M28].",
            "No: Order owns its private data; Reporting must use an API or events [M24].", ""]
valid = []
for q, example in zip(questions, scripted):
    hits = retrieve(q)
    if not hits:
        print("No matching lecture note; no model call:", q)
        continue
    evidence = "\n".join(f"[{sid}] {text}" for sid, text in hits.items())
    answer = ask(f"Use only these notes:\n{evidence}\nQuestion: {q}\nReply briefly and cite the slide ID.", example=example)
    cited = set(re.findall(r"[MD]\d+", answer))
    valid.append(bool(cited) and cited <= set(hits))
    print(answer, "| citation valid:", valid[-1])
assert all(valid), "Cite only retrieved source IDs."

## What you should see
Searchable history retrieves M28; private-table access retrieves M24. The lunch question abstains. Answers cite their retrieved slides. Distances can vary with package/model versions.

## Try it
1. Ask the small order-details lookup from slide 30C. Compare M27 (API composition) with M28 (CQRS).
2. Ask about an order committed just before its producer crashes. Inspect D55, and explain why an outbox is not a global transaction.
3. Put an incorrect answer beside a valid citation. Why does the code accept the citation while a human should reject the answer?

## Takeaway
Retrieval selects evidence; generation uses it; citation validity and factual support are different checks.